# Running Continuous Evals in Production Without Blowing Your Token Budget

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/falcon-ai-page/observe/continuous-evals-budget.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/falcon-ai-page/observe/continuous-evals-budget.ipynb)

| Time | Difficulty |
|------|------------|
| 15 min | Intermediate |

By the end of this cookbook you will have a continuous eval task running on your production project that scores incoming traces for the failure modes you care about (hallucination, tone, policy violations), capped at a token budget you set in advance, with an alert monitor that pings you only when scores drop. Same coverage as scoring 100% of spans with the heaviest judge model, at roughly 10 to 15% of the token cost.

This notebook is the programmatic version of the [docs page](https://docs.futureagi.com/docs/cookbook/observe/continuous-evals-budget). Steps 2 and 3 use the dashboard in the docs; here they're REST API calls so the whole flow runs from one Python script.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- A traced Observe project. If you don't have one, follow [Manual Tracing](https://docs.futureagi.com/docs/cookbook/quickstart/manual-tracing) first.
- Python 3.9+

## Install

Install the FutureAGI evaluation SDK and `requests`.

In [ ]:
%pip install ai-evaluation requests

In [ ]:
import os
os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"


## Step 1: Pick the right judge model for your workload

The judge model is the single biggest cost lever. Future AGI's three Turing tiers price out roughly 10x apart end to end:

- **`turing_flash`**: Latency-optimized, text and image. Use for routine production evals.
- **`turing_small`**: Higher fidelity at moderate cost. Use when `turing_flash` disagrees with your team too often.
- **`turing_large`**: Flagship multimodal accuracy. Use only for high-stakes evals.

Default to `turing_flash` and only escalate per-eval when you have evidence of disagreement. The cell below sanity-checks the verdict and reason length on a representative span across all three tiers, so you can confirm `turing_flash` is good enough before committing it to a continuous task.

In [ ]:
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

inputs = {
    "input": "What is your return policy?",
    "output": (
        "We accept returns of unopened items within 30 days of purchase for a "
        "full refund. After 30 days, we can offer store credit. To start a "
        "return, click 'Start Return' in your account dashboard. Let me know "
        "if you'd like me to walk you through the process."
    ),
}

for model in ["turing_flash", "turing_small", "turing_large"]:
    r = evaluator.evaluate(eval_templates="is_helpful", inputs=inputs, model_name=model)
    print(f"{model:>14} | {r.eval_results[0].output} | reason length: {len(r.eval_results[0].reason)} chars")

If `turing_flash` gives the same verdict as `turing_large` on a sample of 30 to 50 representative spans, use `turing_flash` for the continuous task. Save the heavier models for the eval correction loop where you're calibrating against human judgment, not running at scale.

## Step 2: Find your project and eval configs

Eval tasks reference a project by UUID and one or more eval configs (the eval template + judge model bound to a project) by UUID. The cell below lists what's available so you can pick the IDs for the eval task in the next step.

In [ ]:
import requests

API_KEY = os.environ["FI_API_KEY"]
SECRET_KEY = os.environ["FI_SECRET_KEY"]
BASE = "https://api.futureagi.com"
HEADERS = {"X-Api-Key": API_KEY, "X-Secret-Key": SECRET_KEY}


def unwrap(resp_json):
    """FAGI API wraps everything as {"status": True, "result": ...}. Pull the inner result."""
    if isinstance(resp_json, dict) and "result" in resp_json:
        return resp_json["result"]
    return resp_json


def safe_json(resp):
    """Print status + body snippet on non-JSON responses so we can debug fast."""
    try:
        return resp.json()
    except ValueError:
        print(f"  HTTP {resp.status_code}, non-JSON response:")
        print(f"  {resp.text[:300]}")
        raise


# 1. List Observe projects
resp = requests.get(f"{BASE}/tracer/project/", headers=HEADERS)
projects = unwrap(safe_json(resp))
project_list = projects.get("projects", projects) if isinstance(projects, dict) else projects
print("Projects:")
for p in project_list[:10]:
    print(f"  {p['id']}  {p['name']}")

# 2. Pick one and list eval configs on it
PROJECT_ID = "<paste-a-project-id-from-above>"

resp = requests.get(
    f"{BASE}/tracer/custom-eval-config/list_custom_eval_configs/",
    headers=HEADERS,
    params={"project_id": PROJECT_ID, "filters": "{}"},
)
configs = unwrap(safe_json(resp))
if isinstance(configs, dict):
    config_list = next((v for v in configs.values() if isinstance(v, list)), [])
else:
    config_list = configs
print(f"\nEval configs on {PROJECT_ID}:")
for c in config_list[:10]:
    print(f"  {c['id']}  {c.get('name')}  model={c.get('model')}")

If the project has no eval configs yet, create one first via **Falcon AI → Evals → Add Evaluation** in the dashboard, or via `POST /tracer/custom-eval-config/` (see [Create Custom Evals](https://docs.futureagi.com/docs/evaluation/features/custom)). For routine production checks, set `model` to `turing_flash`.

> **Tip.** You can attach multiple eval configs to one task (one for hallucination, one for tone, one for a custom policy rule). They all run on each sampled span.

## Step 3: Create the cost-aware continuous eval task

This is where all four cost levers come together: judge model (set on the eval config), filters (LLM spans only, agent name match), sampling rate (10% of matching spans), and span limit (5,000 hard cap per run regardless of traffic). Sampling rate ranges from 1.0 to 100.0, span limit from 1 to 1,000,000.

In [ ]:
task_payload = {
    "project": PROJECT_ID,
    "name": "support-bot-quality-continuous",
    "evals": [
        "<paste-a-config-id-from-step-2>",
        # Add more config IDs to run multiple evals per span
    ],
    "run_type": "continuous",       # or "historical" for one-shot
    "sampling_rate": 10,            # 10% of matching spans (1-100)
    "spans_limit": 5000,            # hard cap per run (ignored when run_type=continuous)
    "filters": {
        "observation_type": ["llm"],
        "span_attributes_filters": [
            {"key": "agent.name", "value": "support_bot", "operator": "equals"}
        ],
    },
}

resp = requests.post(f"{BASE}/tracer/eval-task/", headers=HEADERS, json=task_payload)
print(resp.json())
task = unwrap(safe_json(resp))
TASK_ID = task.get("id") if isinstance(task, dict) else None
print(f"\nEval task created: {TASK_ID}")

Expected response shape: `{"id": "<task-uuid>", "name": "...", "status": "pending", "sampling_rate": 10, "spans_limit": 5000, "run_type": "continuous", ...}`. Status moves through `pending` → `running` → `completed` (or stays `running` for continuous tasks). Pause a runaway task with `POST /tracer/eval-task/<id>/pause/`.

The defaults if you skipped fields (100% sampling, 1000 spans) are wrong for production. They exist to make the first run on a small dataset feel responsive, not to scale.

## Step 4: Wire up an alert monitor so you only react to real drops

Without alerts you're paying for evals nobody reads. The monitor below pages on a 15% drop in pass rate vs the prior window, and logs a warning for a 5% drop. `threshold_operator: "less_than"` means lower scores trigger the alert.

In [ ]:
monitor_payload = {
    "name": "support-bot-quality-drop",
    "metric_type": "evaluation_metrics",
    "metric": "<paste-a-config-id-from-step-2>",        # the CustomEvalConfig UUID
    "threshold_type": "percentage_change",              # or "static" for absolute floor
    "threshold_operator": "less_than",                  # alert when score drops
    "critical_threshold_value": 15,                     # 15% drop pages on-call
    "warning_threshold_value": 5,                       # 5% drop logs to dashboard
    "alert_frequency": 60,                              # check every 60 minutes
    "project": PROJECT_ID,
    "notification_emails": ["oncall@your-company.com"],
}

resp = requests.post(f"{BASE}/tracer/user-alerts/", headers=HEADERS, json=monitor_payload)
print(resp.json())
monitor = unwrap(safe_json(resp))
print(f"\nMonitor created: {monitor.get('id') if isinstance(monitor, dict) else monitor}")

With sampling at 10% and `turing_flash` as the judge, a project doing 50,000 daily LLM spans now pays roughly 5,000 eval calls a day on the cheapest judge tier, with alerts wired up so quality drops actually surface. Compare to the naive setup (100% sampling, `turing_large`) at 50,000 calls on the most expensive judge tier per day. Same regression detection. Roughly 10% of the token spend.

> **Check.** Continuous eval task running on your production project, scoped to user-facing LLM spans, sampling at a rate that matches your traffic, scored by the right judge model for the eval, with an alert monitor wired to your on-call rotation. Eval cost capped, regressions still surfaced.

## Explore further

- **[Run Evals on Traces](https://docs.futureagi.com/docs/observe/features/evals)**: Full reference for the continuous and historical eval task UI
- **[Future AGI Models](https://docs.futureagi.com/docs/evaluation/features/futureagi-models)**: Pick the right judge tier: turing_flash, turing_small, turing_large
- **[Alerts & Monitors](https://docs.futureagi.com/docs/observe/features/alerts)**: Threshold-based alerts on eval metrics with email and PagerDuty